# 01. AETHER On-Ramp

This is the fastest runnable introduction to AETHER.

By the end of this notebook you will have:

- started the authenticated pilot HTTP service
- submitted one tiny AETHER document
- asked the kernel what is true right now
- seen the Python SDK talk to the current v1 pilot boundary

In [ ]:
from pathlib import Path
import subprocess
import sys

candidate_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
repo_root = next(
    (
        path
        for path in candidate_roots
        if (path / "python").exists() and (path / "Cargo.toml").exists()
    ),
    None,
)

if repo_root is None:
    repo_root = Path("/content/AETHER")
    if not repo_root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/fyremael/AETHER", str(repo_root)],
            check=True,
        )

python_root = repo_root / "python"
if str(python_root) not in sys.path:
    sys.path.insert(0, str(python_root))

repo_root

In [ ]:
from notebooks.colab_setup import ensure_rust_toolchain, pretty_json, require_service_capabilities, start_pilot_service, stop_http_service
from aether_sdk import AetherClient

ensure_rust_toolchain()
session = start_pilot_service(repo_root)
client = AetherClient(
    session.base_url,
    bearer_token=session.bearer_token,
    namespace=session.namespace,
)
require_service_capabilities(client)

print(f"Pilot namespace: {session.namespace}")
pretty_json(client.health(), title="Pilot health")
pretty_json(client.status(), title="Pilot service status")

## Run The Smallest Useful Program

This document says:

- `task(x)` is a base fact
- `ready(x)` follows directly from `task(x)`
- materialize `ready`
- ask the current-cut query `ready_now`

In [ ]:
document = """
schema v1 {
}

predicates {
  task(Entity)
  ready(Entity)
}

rules {
  ready(x) <- task(x)
}

materialize {
  ready
}

facts {
  task(entity(1))
  task(entity(2))
}

query ready_now {
  current
  goal ready(x)
  keep x
}
"""

response = client.run_document(document)
pretty_json(response, title="Full document run")

In [ ]:
ready_now = client.run_named_query(document, query_name="ready_now")
pretty_json(ready_now, title="Named query: ready_now")
rows = [row["values"] for row in ready_now["rows"]]
print(f"Plain row values for downstream use: {rows}")

## What You Just Saw

AETHER is not only storing facts.
It is accepting a semantic document, compiling its rules, and returning a current answer through the authenticated pilot boundary.

The status payload in the setup cell also showed the operator facts around that answer: namespace, storage backend, token identity, audit path, and service mode.

That is the smallest live loop in the system: state, rule, answer, and an inspectable operating context.

In [ ]:
# Uncomment this cell when you are done with the notebook.
# stop_http_service(session)